# Wildfire Prediction — Comprehensive Flow Notebook

This notebook unifies all logic from the individual wildfire notebooks into a single end-to-end flow:
1. Preliminary Analysis
2. Exploratory Data Analysis (EDA)
3. Preprocessing
4. Classifier Training and Results
5. Regressor Training and Results
6. Clustering Training and Results

**Source notebooks:** `wildfire-eda.ipynb`, `wildfire-rf-svm.ipynb`, `wildfire-intensity-regression.ipynb`, `wildfire_clustering_pipeline.ipynb`


## 1. Preliminary Analysis

### 1.1 Setup

In [ ]:
# ── From wildfire-eda.ipynb ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Consistent color scheme used across all charts
COLOR_FIRE = "#B85042"
COLOR_NOFIRE = "#5D737E"
COLOR_DAY = "#E8A33D"
COLOR_NIGHT = "#1E2761"

# ── From wildfire-rf-svm.ipynb ───────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

# ── From wildfire-intensity-regression.ipynb ─────────────────────────────
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, r2_score
from lightgbm import LGBMRegressor

# ── From wildfire_clustering_pipeline.ipynb ──────────────────────────────
import os
from pathlib import Path
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

plt.rcParams.update({'figure.dpi': 150, 'figure.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})

RANDOM_STATE = 42
# Minimum minority-class proportion below which SMOTE is applied.
# 0.40 means a class must hold at least 40 % of samples for the
# dataset to be considered balanced enough to skip oversampling.
IMBALANCE_THRESHOLD = 0.40

print("Imports loaded.")


### 1.2 Load Dataset

In [ ]:
# Supports both file names used across the source notebooks
repo_root = Path.cwd()
candidates = [repo_root / 'wildfire.csv', repo_root / 'final_dataset.csv']
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        'Could not find wildfire.csv or final_dataset.csv in the working directory.'
    )

df = pd.read_csv(data_path)
print(f'Loaded: {data_path.name}')
print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()


### 1.3 Dataset Structure

In [ ]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print()
print(df.dtypes)


## 2. Exploratory Data Analysis (EDA)

## 2. Dataset Health Check

Before analyzing any patterns, we verify the raw quality of the data.


### 2.1 Shape & Data Types

In [ ]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print()
print(df.dtypes)


### 2.2 Missing Values

In [ ]:
missing = df.isna().sum()
total_missing = missing.sum()

if total_missing == 0:
    print("No missing values across the entire dataset.")
else:
    print(f"Total missing cells: {total_missing}")
    print(missing[missing > 0])


### 2.3 Duplicate Rows

In [ ]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")


**Finding 1 Data Health:** The dataset has zero missing values and zero duplicates. This is unusually clean because the source (Kaggle) has been pre-processed. In a real-world deployment, our pipeline would need to handle missing satellite observations and sensor noise; this is noted in our limitations section.


## 3. Target Variable Analysis

The pipeline has two supervised targets:
- `occured` - binary fire occurrence (for Model 1 Classification)
- `frp` - Fire Radiative Power in megawatts (for Model 2 Regression)


### 3.1 Classification target: `occured`

In [ ]:
counts = df["occured"].value_counts().sort_index()
proportions = df["occured"].value_counts(normalize=True).sort_index().round(4)

print("Counts:")
print(counts)
print("\nProportions:")
print(proportions)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(["No Fire (0)", "Fire (1)"], counts.values,
              color=[COLOR_NOFIRE, COLOR_FIRE])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, v,
            f"{v:,}\n({v/len(df)*100:.1f}%)",
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Class Balance: Fire Occurrence", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of observations")
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()


**Finding 2 - Perfect class balance (50.0% / 50.0%):**

This has direct modeling implications for the Classification team:
- **No need for SMOTE, class weights or undersampling** - the dataset has already been balanced.
- **Accuracy is a valid metric** here (unlike most real-world fire datasets where fires are rare events).
- **Caveat for limitations slide:** this balance is artificial. Real satellite data has a heavy no-fire majority; our model's reported accuracy will not transfer directly to deployment.


### 3.2 Regression target: `frp` (Fire Radiative Power)

In [ ]:
print(df["frp"].describe())
print(f"\nSkewness: {df['frp'].skew():.2f}")
print(f"Rows with FRP == 0: {(df['frp'] == 0).sum()}")
print(f"Rows with FRP  > 0: {(df['frp'] > 0).sum():,}")


In [ ]:
print("FRP statistics grouped by occured:")
df.groupby("occured")["frp"].describe().round(2)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fire_frp = df[df["occured"] == 1]["frp"]

axes[0].hist(fire_frp, bins=80, color=COLOR_FIRE, edgecolor="white")
axes[0].set_title("FRP Distribution (raw) — fire events only", fontweight="bold")
axes[0].set_xlabel("FRP (MW)")
axes[0].set_ylabel("Frequency")

axes[1].hist(np.log1p(fire_frp), bins=60, color=COLOR_FIRE, edgecolor="white")
axes[1].set_title("FRP Distribution (log1p) — fire events only", fontweight="bold")
axes[1].set_xlabel("log(FRP + 1)")
axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()


**Finding 3 - FRP is heavily right-skewed (skew = 6.18):**

- Raw FRP ranges from 0 to 1,056 MW with most mass concentrated below 30 MW.
- A **log1p transformation** produces a near-normal distribution (right chart above).
- **Recommendation for the Regression team:** train on `log1p(frp)` and inverse-transform at inference. Without this, the model will be dominated by a small number of extreme outliers.

**Finding 4 - Unexpected FRP values in no-fire rows:**

The no-fire subset (`occured=0`) still has a non-zero mean FRP of ~9.2 MW. This suggests:
- FRP is a continuous satellite signal, not a clean "zero if no fire" quantity.
- The `occured` label appears to be threshold-based.
- **Recommendation:** Model 2 (Regression) should be trained only on fire-positive rows (`occured == 1`, n = 59,452) to predict intensity given that a fire exists. The pipeline then uses Model 1's probability as the gating signal.


## 4. Feature Analysis

We now look at the 15 predictor variables: 2 spatial (`lat`, `lon`), 1 temporal (`daynight_N`), and 12 weather-derived features.


### 4.1 Geographic distribution

In [ ]:
print(f"Latitude range:  {df['lat'].min():.2f} to {df['lat'].max():.2f}")
print(f"Longitude range: {df['lon'].min():.2f} to {df['lon'].max():.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

no_fire = df[df["occured"] == 0].sample(
    min(5000, (df["occured"] == 0).sum()), random_state=42)
fire = df[df["occured"] == 1]

ax.scatter(no_fire["lon"], no_fire["lat"], s=2, c=COLOR_NOFIRE,
           alpha=0.3, label="No fire (5k sample)")
ax.scatter(fire["lon"], fire["lat"], s=3, c=COLOR_FIRE,
           alpha=0.5, label="Fire")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Global distribution of observations", fontweight="bold")
ax.legend(loc="lower left")
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
plt.tight_layout()
plt.show()


**Finding 5 - Global coverage with known hotspots:**

Observations concentrate in sub-Saharan Africa, the Amazon basin, Southeast Asia, and Australia - all recognized global fire-prone regions.

### 4.2 Day vs. Night observations

In [ ]:
dn = df.groupby("daynight_N")["occured"].agg(["count", "sum", "mean"])
dn["fire_rate_pct"] = dn["mean"] * 100
print(dn)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(["Day (0)", "Night (1)"], dn["fire_rate_pct"].values,
              color=[COLOR_DAY, COLOR_NIGHT])
for b, v in zip(bars, dn["fire_rate_pct"].values):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:.1f}%",
            ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_title("Fire occurrence rate: Day vs Night", fontweight="bold")
ax.set_ylabel("Fire rate (%)")
ax.set_ylim(0, dn["fire_rate_pct"].max() * 1.2)
plt.tight_layout()
plt.show()


**Finding 6 - `daynight_N` is the single strongest predictor:**

Day and night observations have dramatically different fire rates. This will be a top-ranked feature in the Classification model.

### 4.3 Feature distributions: fire vs. no-fire

In [ ]:
features = ["fire_weather_index", "temp_mean", "humidity_min",
            "wind_speed_max", "solar_radiation_mean", "evapotranspiration_total"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flatten(), features):
    data_nofire = df[df["occured"] == 0][feat].dropna()
    data_fire = df[df["occured"] == 1][feat].dropna()
    ax.hist(data_nofire, bins=40, alpha=0.55, color=COLOR_NOFIRE,
            label="No fire", density=True)
    ax.hist(data_fire, bins=40, alpha=0.65, color=COLOR_FIRE,
            label="Fire", density=True)
    ax.set_title(feat, fontweight="bold", fontsize=11)
    ax.legend(fontsize=9)
plt.suptitle("Feature distributions: Fire vs No-Fire",
             fontweight="bold", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()


In [ ]:
feat_cols = ["fire_weather_index", "temp_mean", "humidity_min",
             "wind_speed_max", "solar_radiation_mean", "cloud_cover_mean",
             "dewpoint_mean", "evapotranspiration_total", "temp_range"]
df.groupby("occured")[feat_cols].mean().round(2).T


**Finding 7 - Separation is modest on individual features:**

Fire vs no-fire distributions overlap substantially for every single feature. Mean differences are directionally correct (fires occur at lower humidity, higher FWI, higher evapotranspiration) but small. This strongly suggests:

- **Linear models will underperform** - no single feature cleanly separates the classes.
- **Tree-based models (Random Forest, XGBoost)** should perform better because fire occurrence depends on non-linear interactions between features.


## 5. Correlation Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.75}, ax=ax,
            annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()


### 5.1 Correlation with `occured`

In [ ]:
corr_occured = (df.drop(columns=["frp"])
                   .corr(numeric_only=True)["occured"]
                   .sort_values(key=abs, ascending=False))
print(corr_occured)


### 5.2 Correlation with `frp` (fire-only subset)

In [ ]:
fire_df = df[df["occured"] == 1]
corr_frp = (fire_df.drop(columns=["occured"])
                   .corr(numeric_only=True)["frp"]
                   .sort_values(key=abs, ascending=False))
print(corr_frp)
print(f"\nFire-only subset size: {len(fire_df):,}")


**Finding 8 - Multicollinearity among weather features:**

Several weather features are heavily correlated with each other:
- `evapotranspiration_total` ↔ `fire_weather_index`: **+0.84** (near-duplicate signal)
- `humidity_min` ↔ `dewpoint_mean`: **+0.72**
- `humidity_min` ↔ `temp_range`: **−0.71**
- `humidity_min` ↔ `fire_weather_index`: **−0.65**

FWI is a composite index derived from temperature, humidity, wind, and precipitation - so this collinearity is expected. Recommendations:
- For **tree-based models** (likely what the team will use), multicollinearity is not a problem; models can be trained with the full feature set.
- For any **linear model** baseline, consider dropping `evapotranspiration_total` (redundant with FWI) or using Ridge regression.

**Finding 9 - FRP is hard to predict:**

Even within the fire-only subset, no feature correlates with FRP above 0.11. Fire intensity appears to be governed by factors not captured in this dataset (fuel load, vegetation type, terrain, wind gusts, ignition source). **Expect modest R² on Model 2.** This is a critical point for the limitations section.


## 6. Summary of Findings & Modeling Implications

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Zero missing values, zero duplicates | Real-world deployment will need data-quality handling |
| 2 | Perfect 50/50 class balance on `occured` | No SMOTE needed; accuracy is a valid metric |
| 3 | FRP is heavily right-skewed (skew = 6.18) | Regression team should log-transform the target |
| 4 | Non-zero FRP in no-fire rows | Train Model 2 only on fire-positive rows (n = 59,452) |
| 5 | Global coverage, known fire-prone regions | Used to project Model 3's weather-based clusters onto geography for interpretation |
| 6 | `daynight_N` has the strongest signal (−0.29 corr) | Expect it to be the top feature in Classification |
| 7 | Modest feature-level separation between classes | Prefer tree-based models over linear ones |
| 8 | Multicollinearity between FWI and weather features | Not an issue for trees; matters for linear baselines |
| 9 | All features correlate weakly with FRP (max 0.11) | Expect modest regression R²; flag in limitations |

---

### Decisions locked in by this EDA

1. **Shared preprocessed dataset** — all three models start from this cleaned file (no additional cleaning needed).
2. **Classification** (Model 1) - full dataset, no resampling, tree-based model preferred.
3. **Regression** (Model 2) - fire-positive subset only, log1p-transformed target.
4. **Clustering** (Model 3) - 9 meteorological features as inputs (`lat`, `lon` excluded); k = 4 chosen by domain reasoning (four-tier fire-danger convention). Cluster labels projected onto lat/lon for visualization only.

## 3. Preprocessing

In [ ]:
features = [c for c in df.columns if c not in ['occured', 'frp', 'frp_log']]
X = df[features]
y = df['occured']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

### Apply SMOTE if training set is imbalanced

The source dataset is pre-balanced (50/50), so SMOTE is skipped when the minority class holds at least `IMBALANCE_THRESHOLD` of training samples.


In [ ]:
train_balance = y_train.value_counts(normalize=True)
imbalanced = train_balance.min() < IMBALANCE_THRESHOLD

if imbalanced:
    sm = SMOTE(random_state=42)
    X_res, y_res = sm.fit_resample(X_train, y_train)
    print(f'SMOTE applied. Resampled training size: {len(X_res):,}')
else:
    X_res, y_res = X_train.copy(), y_train.copy()
    print(f'Dataset already balanced ({train_balance.to_dict()}); SMOTE skipped.')

print(f'Training class balance after resampling:\n{pd.Series(y_res).value_counts()}')


## 4. Classifier Training and Results

### 4.1 Random Forest

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_res, y_res)
rf_preds = rf.predict(X_test)
print("RF:", classification_report(y_test, rf_preds))
print("RF AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

#### Feature Importance

In [ ]:
# Feature importance
fi = pd.Series(rf.feature_importances_,
               index=features).sort_values(ascending=False)
fi.head(10).plot(kind='barh').invert_yaxis()
plt.title("Top 10 Features -- Random Forest")
plt.show()

#### Confusion Matrix — Random Forest

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, rf_preds, ax=ax,
                                        display_labels=['No Fire', 'Fire'],
                                        colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()


### 4.2 SVM (RBF Kernel)

In [ ]:
# SVM
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_res)
X_test_sc = scaler.transform(X_test)

svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_sc, y_res)
svm_preds = svm.predict(X_test_sc)
print("SVM:", classification_report(y_test, svm_preds))

#### SVM ROC-AUC

In [ ]:
#roc auc
print("SVM AUC:", roc_auc_score(y_test, svm.predict_proba(X_test_sc)[:, 1]))

#### Confusion Matrix — SVM

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, svm_preds, ax=ax,
                                        display_labels=['No Fire', 'Fire'],
                                        colorbar=False)
ax.set_title('Confusion Matrix — SVM', fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Regressor Training and Results
This notebook trains a predictive model solely on instances where a **fire actually occurred**.

Unlike two-stage models, we skip predicting fire occurrence here and assume we are explicitly modeling the intensity (FRP - Fire Radiative Power) of verified fire events.

### Objectives:
1. Drop all non-fire rows from the dataset before splitting.
2. Train a fast LightGBM regression model on fire events.
3. Evaluate the model strictly on unseen fire events.
4. Visualize Feature Importance and save the model.

### 5.1 Data Filtering & Preparation
We load the data, immediately drop any rows where `occured == 0`, drop the `occured` column completely, and apply a log transformation to the right-skewed `frp` variable.

In [ ]:
# df already loaded in Section 1; filter to fire-only rows
# EXPLICIT FILTER: Keep only rows where a fire occurred
fire_only_df = df[df['occured'] == 1.0].copy()

# Drop the 'occured' column as it has no variance now (all are 1.0)
fire_only_df = fire_only_df.drop(columns=['occured'])

# Separate Features (X) and Target (y)
# We apply a log1p transformation to frp to handle extreme skewness in fire intensity
X = fire_only_df.drop(columns=['frp'])
y = np.log1p(fire_only_df['frp'])

# Train-Test Split (Only on fire data!)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Original dataset shape: {df.shape}")
print(f"Fire-only dataset shape: {fire_only_df.shape}")
print(f"Training set shape: {X_train_reg.shape}")
print(f"Testing set shape: {X_test_reg.shape}")

### 5.2 Model Training
We will use a highly optimized `LGBMRegressor`.

In [ ]:
# Initialize and train the model
reg_model = LGBMRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=50,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

reg_model.fit(X_train_reg, y_train_reg)
print("Model training complete.")

### 5.3 Evaluation & Insights
We evaluate strictly on the testing subset of the **fire-only** data. We also need to reverse the `log1p` transformation to get real-world errors.

In [ ]:
# Predict on the fire-only test set
log_preds = reg_model.predict(X_test_reg)

# Reverse log transformation (expm1 is the inverse of log1p)
y_test_real = np.expm1(y_test_reg)
preds_real = np.expm1(log_preds)

# Metrics
r2 = r2_score(y_test_real, preds_real)
mae = mean_absolute_error(y_test_real, preds_real)

print(f"R-squared (R2): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_test_real, preds_real, alpha=0.4, color='teal')
axes[0].plot([0, max(y_test_real)], [0, max(y_test_real)], 'r--') # 1:1 Reference line
axes[0].set_xlabel('Actual FRP')
axes[0].set_ylabel('Predicted FRP')
axes[0].set_title('Actual vs Predicted Fire Intensity')
axes[0].set_xlim(0, np.percentile(y_test_real, 95)) # Cap at 95th percentile to hide extreme outliers
axes[0].set_ylim(0, np.percentile(preds_real, 95))
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. Residuals
residuals = y_test_real - preds_real
axes[1].scatter(preds_real, residuals, alpha=0.4, color='darkred')
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_xlabel('Predicted FRP')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title('Residual Plot')
axes[1].set_xlim(0, np.percentile(preds_real, 95))
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### 5.4 Feature Importance
What factors drive the intensity of a fire once it has already started?

In [ ]:
importance = reg_model.feature_importances_
features = X_train_reg.columns

fi_df = pd.DataFrame({'Feature': features, 'Importance': importance})
fi_df = fi_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=fi_df, palette='magma')
plt.title('Feature Importance for Fire Intensity (LightGBM)')
plt.xlabel('Importance (Tree Splits)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### 5.5 Save Model
Export the regression model for downstream systems.

In [ ]:
joblib.dump(reg_model, 'direct_fire_intensity_lgbm.pkl')
print("Model saved successfully as 'direct_fire_intensity_lgbm.pkl'")

## 6. Clustering Training and Results
K-Means baseline and GMM main model on meteorological features only. Fire-weather index, occurred, and frp are used only for post-hoc validation. Lat and lon is only used to plot data points onto map.


In [ ]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 150, 'figure.facecolor': 'white', 'axes.spines.top': False, 'axes.spines.right': False})

RANDOM_STATE = 42
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

CLUSTER_FEATURES = [
    'pressure_mean', 'wind_direction_mean', 'wind_direction_std',
    'solar_radiation_mean', 'dewpoint_mean', 'cloud_cover_mean',
    'evapotranspiration_total', 'humidity_min', 'temp_mean',
    'temp_range', 'wind_speed_max'
]
VALIDATION_FEATURES = ['fire_weather_index', 'lat', 'lon', 'occured', 'frp']
geo_colors = ['#2166ac', '#67a9cf', '#fdae61', '#b2182b']
RISK_TIER_NAMES = ['Low', 'Mod', 'High', 'Extreme']


def remap_labels_by_fwi(frame: pd.DataFrame, label_col: str) -> dict:
    means = frame.groupby(label_col)['fire_weather_index'].mean().sort_values()
    return {old_label: new_label for new_label, old_label in enumerate(means.index)}


def boxplot_fwi(frame: pd.DataFrame, label_col: str, title: str, output_name: str) -> None:
    ordered_labels = [0, 1, 2, 3]
    values = [frame.loc[frame[label_col] == label, 'fire_weather_index'].to_numpy() for label in ordered_labels]

    fig, ax = plt.subplots(figsize=(8, 5))
    bp = ax.boxplot(values, labels=RISK_TIER_NAMES, patch_artist=True, showfliers=False)
    box_colors = ['#dbe9f6', '#a6bddb', '#fdcc8a', '#e34a33']

    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')

    for artist_group in ['whiskers', 'caps', 'medians']:
        for artist in bp[artist_group]:
            artist.set_color('black')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Risk tier')
    ax.set_ylabel('fire_weather_index')
    ax.grid(True, axis='y', alpha=0.25)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, output_name), bbox_inches='tight', dpi=200)
    plt.show()
    plt.close(fig)


def plot_geo(frame: pd.DataFrame, title: str, output_name: str, lon_min: float, lon_max: float, lat_min: float, lat_max: float) -> None:
    fig, ax = plt.subplots(figsize=(18, 9) if 'Global' in title else (8, 10))
    for label, color in enumerate(geo_colors):
        subset = frame[frame['gmm_label'] == label]
        if subset.empty:
            continue
        ax.scatter(
            subset['lon'],
            subset['lat'],
            s=3 if 'Global' in title else 8,
            alpha=0.45 if 'Global' in title else 0.55,
            color=color,
            linewidths=0,
            label=f'{RISK_TIER_NAMES[label]} (n={len(subset):,})'
        )

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.grid(True, alpha=0.2 if 'Global' in title else 0.25)
    ax.legend(title='Risk tier', markerscale=4 if 'Global' in title else 3, framealpha=0.9)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, output_name), bbox_inches='tight', dpi=200)
    plt.show()
    plt.close(fig)


# df already loaded in Section 1

X_clust = df[CLUSTER_FEATURES].copy()
X_val = df[VALIDATION_FEATURES].copy()

print('Dataset shape:', df.shape)
print('X_clust shape:', X_clust.shape)
print('X_val shape:', X_val.shape)
print('Validation columns held out from fitting:', list(X_val.columns))

scaler = StandardScaler()
X_arr = scaler.fit_transform(X_clust)
joblib.dump(scaler, 'scaler.pkl')
print('Saved scaler -> scaler.pkl')


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
kmeans_raw = kmeans.fit_predict(X_arr)
kmeans_map = remap_labels_by_fwi(df.assign(kmeans_label=kmeans_raw), 'kmeans_label')
df['kmeans_label'] = pd.Series(kmeans_raw, index=df.index).map(kmeans_map).astype(int)
kmeans_means = df.groupby('kmeans_label')['fire_weather_index'].mean().sort_index()
kmeans_silhouette = silhouette_score(X_arr, df['kmeans_label'], sample_size=min(10000, len(df)), random_state=RANDOM_STATE)

print(f'K-Means silhouette_score (K=4): {kmeans_silhouette:.4f}')
print('K-Means mean fire_weather_index by cluster:')
display(kmeans_means.rename('mean_fire_weather_index').to_frame())

k_values = [2, 3, 4, 5, 6]
sil_scores = []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_arr)
    sil_scores.append(silhouette_score(X_arr, labels, sample_size=min(10000, len(df)), random_state=RANDOM_STATE))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, sil_scores, marker='o', linewidth=2, color='#1f77b4')
ax.set_xticks(k_values)
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette Sweep for K-Means')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'silhouette_sweep.png'), bbox_inches='tight', dpi=200)
plt.show()
plt.close(fig)

joblib.dump(kmeans, 'kmeans_model.pkl')
print('Saved model -> kmeans_model.pkl')


In [ ]:
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=RANDOM_STATE)
gmm.fit(X_arr)
gmm_raw = gmm.predict(X_arr)
gmm_proba = gmm.predict_proba(X_arr)
gmm_map = remap_labels_by_fwi(df.assign(gmm_label=gmm_raw), 'gmm_label')
df['gmm_label'] = pd.Series(gmm_raw, index=df.index).map(gmm_map).astype(int)
gmm_means = df.groupby('gmm_label')['fire_weather_index'].mean().sort_index()
gmm_bic = gmm.bic(X_arr)
gmm_aic = gmm.aic(X_arr)

print(f'GMM BIC (K=4): {gmm_bic:.4f}')
print(f'GMM AIC (K=4): {gmm_aic:.4f}')
print('GMM mean fire_weather_index by cluster:')
display(gmm_means.rename('mean_fire_weather_index').to_frame())

bic_scores = []
aic_scores = []
for k in k_values:
    model = GaussianMixture(n_components=k, covariance_type='full', random_state=RANDOM_STATE)
    model.fit(X_arr)
    bic_scores.append(model.bic(X_arr))
    aic_scores.append(model.aic(X_arr))

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(k_values, bic_scores, marker='o', linewidth=2, color='#2ca02c', label='BIC')
ax.plot(k_values, aic_scores, marker='o', linewidth=2, color='#d62728', label='AIC')
ax.set_xticks(k_values)
ax.set_xlabel('Number of components')
ax.set_ylabel('Score (lower is better)')
ax.set_title('GMM BIC/AIC Sweep')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'bic_aic_sweep.png'), bbox_inches='tight', dpi=200)
plt.show()
plt.close(fig)

joblib.dump(gmm, 'gmm_model.pkl')
np.save('gmm_proba.npy', gmm_proba)
print('Saved model -> gmm_model.pkl')
print('Saved probabilities -> gmm_proba.npy')


In [ ]:
# Consolidated post-model analysis and exports
os.makedirs(FIGURES_DIR, exist_ok=True)

for label_col, model_name in [('kmeans_label', 'KMeans'), ('gmm_label', 'GMM')]:
    table = df.groupby(label_col)['fire_weather_index'].mean().sort_index().to_frame('mean_fire_weather_index')
    spearman_corr = spearmanr(table.index.to_numpy(), table['mean_fire_weather_index'].to_numpy()).statistic

    print(f'=== {model_name} mean fire_weather_index by cluster ===')
    display(table)
    print(f'Spearman correlation between cluster rank and mean FWI: {spearman_corr:.4f}')

    boxplot_fwi(
        df,
        label_col,
        f'FWI Distribution by Cluster ({model_name}, K=4)',
        f'fwi_boxplot_{model_name.lower()}.png'
    )

cluster_profile_gmm = df.groupby('gmm_label')[CLUSTER_FEATURES].mean().sort_index()
cluster_profile_kmeans = df.groupby('kmeans_label')[CLUSTER_FEATURES].mean().sort_index()

fire_table = df.groupby('gmm_label').agg(
    fire_occurrence_rate=('occured', 'mean'),
    mean_frp_given_fire=('frp', lambda s: s[s > 0].mean())
).sort_index()

cluster_summary = df.groupby('gmm_label').agg(
    size=('gmm_label', 'count'),
    fire_rate=('occured', 'mean'),
    median_frp=('frp', lambda s: s[s > 0].median())
).sort_index()
min_med_frp = cluster_summary['median_frp'].min()
cluster_summary['zone_multiplier'] = (cluster_summary['median_frp'] / min_med_frp).round(2)
cluster_summary.index.name = 'cluster_label'

display(cluster_profile_gmm)
display(cluster_profile_kmeans)
display(fire_table)
display(cluster_summary)

cluster_profile_gmm.to_csv(os.path.join(FIGURES_DIR, 'cluster_profile_gmm.csv'))
cluster_profile_kmeans.to_csv(os.path.join(FIGURES_DIR, 'cluster_profile_kmeans.csv'))
fire_table.to_csv(os.path.join(FIGURES_DIR, 'fire_occurrence_frp_by_gmm.csv'))
cluster_summary.to_csv(os.path.join(FIGURES_DIR, 'cluster_summary_gmm.csv'))
df.to_csv('df_with_labels.csv', index=False)

joblib.dump(scaler, 'scaler.pkl')
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(gmm, 'gmm_model.pkl')
np.save('gmm_proba.npy', gmm_proba)

print('Saved outputs:')
print('  scaler.pkl')
print('  kmeans_model.pkl')
print('  gmm_model.pkl')
print('  gmm_proba.npy')
print('  df_with_labels.csv')
print(f"  {os.path.join(FIGURES_DIR, 'cluster_profile_gmm.csv')}")
print(f"  {os.path.join(FIGURES_DIR, 'cluster_profile_kmeans.csv')}")
print(f"  {os.path.join(FIGURES_DIR, 'fire_occurrence_frp_by_gmm.csv')}")
print(f"  {os.path.join(FIGURES_DIR, 'cluster_summary_gmm.csv')}")

In [ ]:
# Consolidated visualization suite
from sklearn.decomposition import PCA

_TIER_XTICKLABELS = ['Low', 'Mod', 'High', 'Extreme']
_CLUSTER_COLORS = ['#2166ac', '#67a9cf', '#fdae61', '#b2182b']
_BAR_COLORS = ['#dbe9f6', '#a6bddb', '#fdcc8a', '#e34a33']
_CLUSTER_LABELS = [0, 1, 2, 3]

# Feature means by cluster (selected variables)
_PANEL_FEATURES = [
    ('temp_mean', 'Mean Temperature (deg C)'),
    ('humidity_min', 'Minimum Humidity (%)'),
    ('wind_speed_max', 'Max Wind Speed (m/s)'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (feat, ylabel) in zip(axes, _PANEL_FEATURES):
    means = df.groupby('gmm_label')[feat].mean()
    stds = df.groupby('gmm_label')[feat].std()
    ax.bar(
        _CLUSTER_LABELS,
        means,
        yerr=stds,
        color=_CLUSTER_COLORS,
        edgecolor='black',
        linewidth=0.6,
        capsize=4,
        error_kw={'linewidth': 1}
    )
    ax.set_xticks(_CLUSTER_LABELS)
    ax.set_xticklabels(_TIER_XTICKLABELS, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.grid(True, axis='y', alpha=0.25)

fig.suptitle('Feature Means by GMM Cluster (error bars = +/-1 std)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_feature_means_3panel.png'), bbox_inches='tight', dpi=200)
plt.show()
plt.close(fig)

# Standardized heatmap (GMM profile)
gmm_profile_z = (cluster_profile_gmm - cluster_profile_gmm.mean(axis=0)) / cluster_profile_gmm.std(axis=0, ddof=0)
fig, ax = plt.subplots(figsize=(14, 4.5))
img = ax.imshow(gmm_profile_z.to_numpy(), cmap='RdYlBu_r', aspect='auto')
for row_idx in range(gmm_profile_z.shape[0]):
    for col_idx in range(gmm_profile_z.shape[1]):
        value = gmm_profile_z.iloc[row_idx, col_idx]
        text_color = 'white' if abs(value) >= 0.6 else 'black'
        ax.text(col_idx, row_idx, f'{value:.2f}', ha='center', va='center', fontsize=8, color=text_color)
ax.set_xticks(np.arange(len(CLUSTER_FEATURES)))
ax.set_xticklabels(CLUSTER_FEATURES, rotation=45, ha='right')
ax.set_yticks(np.arange(len(gmm_profile_z.index)))
ax.set_yticklabels([RISK_TIER_NAMES[int(k)] for k in gmm_profile_z.index])
ax.set_title('Standardized Cluster Profile Heatmap (GMM, K=4)', fontweight='bold')
cb = fig.colorbar(img, ax=ax)
cb.set_label('Z-score (within feature)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cluster_profile_heatmap_gmm_zscore.png'), bbox_inches='tight', dpi=200)
plt.show()
plt.close(fig)

# FRP and zone multiplier
_baseline_idx = int(cluster_summary['median_frp'].idxmin())
_tier_names = ['Low', 'Mod', 'High', 'Extreme']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(_CLUSTER_LABELS, cluster_summary['median_frp'], color=_BAR_COLORS, edgecolor='black')
for i, value in enumerate(cluster_summary['median_frp']):
    axes[0].annotate(f'{value:.2f}', xy=(i, value), xytext=(0, 4), textcoords='offset points', ha='center', va='bottom', fontsize=10)
axes[0].set_xticks(_CLUSTER_LABELS)
axes[0].set_xticklabels(_TIER_XTICKLABELS)
axes[0].set_ylabel('Median FRP (MW, fire pixels only)')
axes[0].set_title('Median FRP by GMM Cluster', fontweight='bold')
axes[0].grid(True, axis='y', alpha=0.25)

axes[1].bar(_CLUSTER_LABELS, cluster_summary['zone_multiplier'], color=_BAR_COLORS, edgecolor='black')
for i, zm in enumerate(cluster_summary['zone_multiplier']):
    axes[1].text(i, zm + 0.01, f'{zm:.2f}', ha='center', va='bottom', fontsize=10)
axes[1].set_xticks(_CLUSTER_LABELS)
axes[1].set_xticklabels(_TIER_XTICKLABELS)
axes[1].set_ylabel('Zone Multiplier')
axes[1].set_title('Zone Multiplier by GMM Cluster', fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.25)
axes[1].text(
    0.98,
    0.02,
    f"Baseline = {_tier_names[_baseline_idx]} (1.00)",
    transform=axes[1].transAxes,
    ha='right',
    va='bottom',
    fontsize=9
 )

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_frp_zonemult.png'), bbox_inches='tight', dpi=200)
plt.show()
plt.close(fig)

# Silhouette and BIC/AIC with K=4 annotations
_score_at_k4 = sil_scores[k_values.index(4)]
_score_at_k2 = sil_scores[k_values.index(2)]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, sil_scores, marker='o', linewidth=2, color='#1f77b4')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.7)
ax.annotate('Domain-selected\n(4-tier convention)', xy=(4, _score_at_k4), xytext=(4.2, _score_at_k4 + 0.005), arrowprops=dict(arrowstyle='->'), fontsize=9)
ax.annotate('K=2: binary split', xy=(2, _score_at_k2), xytext=(2.2, _score_at_k2 - 0.008), fontsize=9)
ax.set_xticks(k_values)
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette Sweep for K-Means', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_silhouette_annotated.png'), bbox_inches='tight', dpi=200)
plt.show()
plt.close(fig)

_bic_at_k4 = bic_scores[k_values.index(4)]
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(k_values, bic_scores, marker='o', linewidth=2, color='#2ca02c', label='BIC')
ax.plot(k_values, aic_scores, marker='o', linewidth=2, color='#d62728', label='AIC')
ax.axvline(x=4, color='orange', linestyle='--', alpha=0.8, label='Domain-selected K')
ax.annotate('Marginal gain flattens near K=4', xy=(4, _bic_at_k4), xytext=(4.2, _bic_at_k4 + 5000), arrowprops=dict(arrowstyle='->'), fontsize=9)
ax.set_xticks(k_values)
ax.set_xlabel('Number of components')
ax.set_ylabel('Score (lower is better)')
ax.set_title('GMM BIC/AIC Sweep', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_bic_aic_annotated.png'), bbox_inches='tight', dpi=200)
plt.show()
plt.close(fig)


### K=4 Model Selection Justification

We choose **K=4** primarily by domain convention: wildfire danger communication is commonly framed as a four-tier risk scale (very low, low, moderate, high).

The monotonic decrease in GMM **BIC/AIC** with larger K is expected for meteorological data with continuous gradients, where additional components keep fitting finer structure.

The relatively low silhouette values are also expected for this feature space because environmental transitions are gradual rather than sharply separated; therefore, silhouette is informative but not the sole decision criterion.

### Spearman Rho Interpretation Note

The observed Spearman \(\rho = 1.0\) here is guaranteed by construction of `remap_labels_by_fwi`, which explicitly reorders cluster labels by ascending mean FWI.

Interpret this as a **sanity check that remapping worked as intended**, not as independent external validation of clustering quality.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# --- Resolve cluster label column from previous cells ---
cluster_candidates = ["cluster", "gmm_label", "kmeans_label"]
cluster_col = next((c for c in cluster_candidates if c in df.columns), None)
if cluster_col is None:
    raise ValueError(
        "No cluster label column found. Expected one of: "
        f"{cluster_candidates}. Available columns include: {list(df.columns)[:20]}..."
    )
print(f"Using cluster column: {cluster_col}")

# --- Validate required columns ---
required_cols = ["frp", "lat", "lon", "occured", cluster_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Work on a clean subset to avoid NaN issues in plots
invest_df = df[required_cols].copy()
invest_df = invest_df.rename(columns={cluster_col: "cluster"})
invest_df = invest_df.dropna(subset=["frp", "lat", "lon", "cluster"])

# Ensure numeric/dtype consistency
invest_df["cluster"] = pd.to_numeric(invest_df["cluster"], errors="coerce")
invest_df = invest_df.dropna(subset=["cluster"])
invest_df["cluster"] = invest_df["cluster"].astype(int)

# Restrict to expected cluster labels
cluster_order = [0, 1, 2, 3]
invest_df = invest_df[invest_df["cluster"].isin(cluster_order)]

# 1) Cluster Distribution (absolute counts + percentages)
cluster_counts = invest_df["cluster"].value_counts().reindex(cluster_order, fill_value=0)
cluster_pct = (cluster_counts / cluster_counts.sum() * 100).round(2)
cluster_distribution = pd.DataFrame(
    {
        "count": cluster_counts,
        "percentage": cluster_pct,
    }
)
print("\nCluster distribution (count and %):")
print(cluster_distribution)

# 2) Outlier Investigation (FRP distributions by cluster)
# Filter FRP <= 0 so the log y-axis is valid and spread is visible.
plot_df = invest_df[invest_df["frp"] > 0].copy()

plt.figure(figsize=(14, 8))
sns.boxplot(
    data=plot_df,
    x="cluster",
    y="frp",
    order=cluster_order,
    palette="Set2",
    showfliers=False,
)
plt.yscale("log")
plt.title("FRP Distribution by Cluster (log scale)")
plt.xlabel("Cluster")
plt.ylabel("FRP (log scale)")
plt.tight_layout()
plt.show()

# 3) Geographic Context (lon-lat scatter, colored by cluster)
colors = sns.color_palette("tab10", n_colors=len(cluster_order))
color_map = dict(zip(cluster_order, colors))

fig, ax = plt.subplots(figsize=(12, 8))
for c in cluster_order:
    grp = invest_df[invest_df["cluster"] == c]
    ax.scatter(
        grp["lon"],
        grp["lat"],
        s=1,
        alpha=0.2,
        color=color_map[c],
        label=f"Cluster {c}",
    )

ax.set_title("Geographic Distribution of Clusters")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

legend_handles = [
    Line2D([0], [0], marker="o", color="w", label=f"Cluster {c}", markerfacecolor=color_map[c], markersize=8)
    for c in cluster_order
]
ax.legend(handles=legend_handles, title="Cluster", loc="best")
plt.tight_layout()
plt.show()